# Topical Lectures, Controls
## Extra: calibrating the motors of the real drone

In `2Dsim_02_system_identification` you found three things about the simulated drone:

1. **`V_left_offset`**: how much more the left motor needs than the right one so that the drone does not roll,
2. **`V_hover`**: the throttle at which the thrust carries the weight, found where `z'' = 0`,
3. **`V_min`, `V_max`**: the range of throttle the motors follow before they clip.

This notebook finds the same numbers for your real Crazyflie, all the way from switching it on.

### Why not ramp the throttle as in the simulator?

In the simulator we ramped the voltage and looked for the moment the acceleration crossed zero. On the real drone a ramp of the raw thrust without any control is how drones end up in the ceiling: the moment `z''` crosses zero the drone is already climbing, with nothing to stop it.

We let the firmware do the ramp for us instead. Its controllers hold the drone in a steady hover, and in a steady hover `z'' = 0` and `φ'' = 0` **by definition**. The motor commands the firmware needs to get there are exactly what we are looking for:

* the **mean** of the four motor commands is `V_hover`,
* the **difference** between the left and the right motors is `V_left_offset`. The integrators of the firmware's attitude loop find this trim by themselves, and we read it off the motors. The real drone has four motors, so we also get the front/back offset and the yaw offset, which the 2D simulator does not have.

The drone logs its four motor commands, `motor.m1` to `motor.m4`, as 16-bit PWM values from 0 to 65535. We divide by 65535 to get the throttle `V` between 0 and 1, the convention of the simulator.

### The layout of the motors

Seen from above, with the forward side up (the blue LEDs at the back):

```
        forward
   M4 (front-left)   M1 (front-right)
            \           /
             \   CF    /
             /         \
            /           \
   M3 (back-left)    M2 (back-right)
```

M1 and M3 spin one way, M2 and M4 the other. More thrust on the left pair (M3, M4) rolls the drone to the right, more on the front pair (M1, M4) pitches it backwards, and more on M1 and M3 against M2 and M4 yaws it.

### Safety

* Fly only in the marked area. One person at the keyboard, one watching the drone, nobody within arm's length while the motors run.
* The person at the keyboard keeps the abort window in front, clicked once so that it takes the keys. **Enter** aborts and lands.
* The flight lands by itself when the drone leaves a 1.5 m box around the start, when no state arrives any more, when the battery drops below 3.1 V or when the flow deck stops delivering.
* Full battery: above 3.5 V at rest. Fly the calibration with the setup you will fly with afterwards, cage included: the cage adds weight and changes `V_hover`.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import crazyflie

%matplotlib inline

# The label of your drone as it reads on the sticker:
# LABEL = 2, or LABEL = 'A' for a letter.
LABEL = 4
URI = crazyflie.drone_uri(LABEL)
print(URI)

PWM_MAX = 65535      # full throttle, V = 1
F_MAX = 0.148        # N, maximal thrust of one real motor, as in the simulator (drone_parameters.md)
G = 9.8              # m/s^2, as in the simulator

## Step 1: find the drone

Switch the drone on (blue LEDs), plug in the Crazyradio dongle. The cell looks for your drone on all channels. A warning means the drone is not on the channel of its label: ask an instructor before you go on.

In [ ]:
crazyflie.find_drones(LABEL)

## Step 2: connect and check

Put the drone **on the floor of the flying area**. The cell prints the battery, whether the flow deck is detected, the supervisor flags and whether the position estimate stays still on the ground, then blinks the LEDs. Make sure **your** drone blinks.

* `flow deck: NOT attached`: reseat the deck.
* `is locked: yes`: run the cell after this one.
* `WARNING: the position estimate runs away` or `does not follow the height sensor`: switch the drone off and on and run this cell again.

In [ ]:
with crazyflie.connect(URI) as scf:
    crazyflie.status(scf)
    crazyflie.identify(scf)

In [ ]:
# Only when the status says locked: clear the lock, then check again.
with crazyflie.connect(URI) as scf:
    crazyflie.recover(scf)
    crazyflie.status(scf)

## Step 3: test the abort

Once per session. A dummy loop of 10 s opens the same abort window as the flight. Click into the window and press Enter: the loop must report the abort. If it does not, do not fly from this program.

In [ ]:
crazyflie.bench_test_abort()

## Step 4: the calibration flight

One flight, nothing but a hover: take off to 0.5 m, hover for `HOLD_S` seconds, land. The firmware holds the drone still; the log records the four motor commands and the battery voltage twenty times a second.

Put the drone flat on the floor in the middle of the flying area, forward side into the free space. Announce the flight and run the cell. Enter in the abort window lands the drone at any point.

The longer the hover, the better the average, but the battery drops while the drone flies, and with it the voltage the motors get. 10 s is a good start.

In [ ]:
HOLD_S = 10.0
Z_HOVER = 0.5
RUN_LABEL = 'motor_cal'

run = crazyflie.fly_path([('hold', HOLD_S)], uri=URI, z=Z_HOVER)
name = crazyflie.save(run, RUN_LABEL)

## Step 5: what the motors did

The first plot shows the whole flight: the four motor commands as throttle `V`, the height and the battery. The take-off and the landing show as the large swings at the start and the end. The grey band is the steady hover that the calibration uses: the hold, minus its first second, while the drone settles.

Look at the plot before you trust the numbers: is the hover in the grey band really steady? Do the four lines sit apart from each other by a constant amount?

In [ ]:
def hover_window(name, trim_s=1.0):
    # The loaded run and a mask over its power samples for the steady hover
    r = crazyflie.load_run(name)
    pid, power = r['pid'], r['power']
    hold = pid['t_ms'][pid['step'] > 0.5]
    if len(hold) == 0:
        raise ValueError('this run has no hover phase, it was aborted before the hold')
    t0, t1 = hold[0] + 1000 * trim_s, hold[-1]
    window = (power['t_ms'] >= t0) & (power['t_ms'] <= t1)
    if window.sum() < 20:
        raise ValueError('only %d samples in the hover, fly a longer HOLD_S' % window.sum())
    return r, window, (t0, t1)

r, window, (t0, t1) = hover_window(name)
power, state = r['power'], r['state']
V = {m: power['motor.m%d' % m] / PWM_MAX for m in (1, 2, 3, 4)}
tp = (power['t_ms'] - power['t_ms'][0]) / 1000
ts = (state['t_ms'] - power['t_ms'][0]) / 1000

fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1, 1]})
for m, colour in zip((1, 2, 3, 4), ('C0', 'C1', 'C2', 'C3')):
    ax[0].plot(tp, V[m], colour, lw=0.8, label='M%d' % m)
ax[0].set_ylabel('throttle V = PWM / 65535')
ax[0].legend(loc='upper right', ncol=4)
ax[1].plot(ts, state['stateEstimate.z'], 'k')
ax[1].set_ylabel('height [m]')
ax[2].plot(tp, power['pm.vbat'], 'C4')
ax[2].set_ylabel('battery [V]')
ax[2].set_xlabel('t [s]')
for a in ax:
    a.axvspan((t0 - power['t_ms'][0]) / 1000, (t1 - power['t_ms'][0]) / 1000, color='0.9')
    a.grid()
plt.tight_layout()

## Step 6: `V_hover`

In the steady hover the thrust carries the weight, `z'' = 0`, so the mean throttle of the four motors is `V_hover`, the same number the ramp gave you in the simulator.

The simulator turns throttle into thrust with a straight line, `F = F_max · V` per motor. With that line, four motors at `V_hover` carry

$$m\,g = 4\,F_\text{max}\,V_\text{hover} \quad\Rightarrow\quad m = \frac{4\,F_\text{max}\,V_\text{hover}}{g} .$$

The cell compares the mass this gives with the 32 g of a bare Crazyflie 2.1, and the `V_hover` of the simulator, `M g / (4 F_max)` for the same 32 g. Weigh your drone, cage and all, if a scale is around. Real propellers are not that linear: the thrust grows faster than the PWM, roughly with its square, so the straight line overestimates the mass at a high throttle. Keep the mass as a rough check, not a measurement.

In [ ]:
V_all = np.mean([V[m][window] for m in (1, 2, 3, 4)], axis=0)    # mean of the four motors, sample by sample
V_HOVER = float(np.mean(V_all))
V_hover_sim = 0.032 * G / (4 * F_MAX)
mass_linear = 4 * F_MAX * V_HOVER / G

print(f'V_hover = {V_HOVER:.3f}  (PWM {V_HOVER * PWM_MAX:.0f} of {PWM_MAX}), spread during the hover {np.std(V_all):.3f}')
print(f'the simulator, bare 32 g drone: V_hover = {V_hover_sim:.3f}')
print(f'mass from the linear motor model: {1000 * mass_linear:.0f} g  (bare Crazyflie 2.1: 32 g)')

## Step 7: the offsets

Each motor's throttle minus the mean of all four is what the firmware's trim had to add to that motor. From the four we form the three offsets, in the convention of the simulator, `V_left = V_hover + V_left_offset`, `V_right = V_hover`:

* `V_left_offset`: the left pair (M3, M4) minus the right pair (M1, M2). This is the number of `2Dsim_02`.
* `V_front_offset`: the front pair (M1, M4) minus the back pair (M2, M3). A battery or a deck that sits off-centre shows up here and in the left offset: the centre of gravity is not under the middle of the motors.
* `V_yaw_offset`: M1 and M3 minus M2 and M4. The two pairs spin opposite ways; a difference means one pair needs more to cancel the other's drag torque, a sign of a worn or a different propeller or motor.

The offsets in the simulator were around ±0.01. The cell also gives each offset relative to `V_hover`, the scatter of the samples, and the uncertainty of the mean, the scatter divided by the square root of the number of samples. An offset more than three times its uncertainty stands above the noise. The samples of a hover are not independent, the drone sways slowly, so the true uncertainty is somewhat larger: the repeat flight in the questions is the honest test.

In [ ]:
means = {m: float(np.mean(V[m][window])) for m in (1, 2, 3, 4)}
for m in (1, 2, 3, 4):
    print(f'M{m}: V = {means[m]:.4f}, {means[m] - V_HOVER:+.4f} from the mean')

def pair_difference(a, b):
    # mean of pair a minus mean of pair b: its mean, the scatter of the samples and the uncertainty of the mean
    d = (V[a[0]] + V[a[1]] - V[b[0]] - V[b[1]])[window] / 2
    return float(np.mean(d)), float(np.std(d)), float(np.std(d) / np.sqrt(len(d)))

V_LEFT_OFFSET, s_left, e_left = pair_difference((3, 4), (1, 2))
V_FRONT_OFFSET, s_front, e_front = pair_difference((1, 4), (2, 3))
V_YAW_OFFSET, s_yaw, e_yaw = pair_difference((1, 3), (2, 4))
print()
for label, value, scatter, error in (('V_left_offset ', V_LEFT_OFFSET, s_left, e_left),
                                     ('V_front_offset', V_FRONT_OFFSET, s_front, e_front),
                                     ('V_yaw_offset  ', V_YAW_OFFSET, s_yaw, e_yaw)):
    print(f'{label} = {value:+.4f} +- {error:.4f}  ({100 * value / V_HOVER:+.1f} % of V_hover), '
          f'scatter of the samples {scatter:.4f}' + ('' if abs(value) > 3 * error else '  (not above the noise)'))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].bar(['M1\nfront-right', 'M2\nback-right', 'M3\nback-left', 'M4\nfront-left'],
          [means[m] - V_HOVER for m in (1, 2, 3, 4)], color=['C0', 'C1', 'C2', 'C3'])
ax[0].axhline(0, color='k', lw=0.8)
ax[0].set_ylabel('V - V_hover')
ax[0].set_title('each motor against the mean')
ax[0].grid(axis='y')
ax[1].bar(['left', 'front', 'yaw'], [V_LEFT_OFFSET, V_FRONT_OFFSET, V_YAW_OFFSET],
          yerr=[e_left, e_front, e_yaw], capsize=6, color='0.6')
ax[1].axhline(0, color='k', lw=0.8)
ax[1].set_title('the three offsets, with the uncertainty of the mean')
ax[1].grid(axis='y')
plt.tight_layout()

## Step 8: the range and the battery

**The range.** The throttle runs from 0 to 1, PWM 0 to 65535, as `V_min = 0` and `V_max = 1` in the simulator; the firmware clips its motor commands there. We do not fly into the clip to find it, as in the simulator: what matters for a controller is how much is left above the hover. The cell gives the headroom, `1 - V_hover`, the highest throttle any motor reached during the whole flight, take-off included, and the thrust-to-weight ratio of the linear model, `1 / V_hover`. Below about 1.5 the drone has little left for tilting and climbing at the same time.

**The battery.** While the drone hovers, the battery voltage drops. With the same PWM, a lower voltage gives the motors less thrust, so the hover throttle creeps up during the flight, unless the firmware compensates for the battery. The right plot shows the mean throttle against the battery voltage during the hover, and the slope of a straight line through it. A flat line means the firmware compensates; a slope means `V_hover` depends on the charge, and a calibration on a full battery is too low for a half-empty one.

In [ ]:
V_peak = max(float(np.max(V[m])) for m in (1, 2, 3, 4))
print(f'V_hover {V_HOVER:.3f}: headroom {1 - V_HOVER:.3f} up to V_max = 1, thrust-to-weight about {1 / V_HOVER:.2f} (linear model)')
print(f'highest throttle of any motor during the flight: {V_peak:.3f}' + ('  CLIPPED' if V_peak >= 0.999 else ''))

vbat = power['pm.vbat'][window]
slope, intercept = np.polyfit(vbat, V_all, 1) if np.ptp(vbat) > 0.01 else (0.0, V_HOVER)
print(f'battery during the hover {vbat.max():.2f} V -> {vbat.min():.2f} V; '
      f'V_hover changes by {slope:+.3f} per volt' + ('  (the voltage hardly changed)' if np.ptp(vbat) <= 0.01 else ''))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(V_all, bins=30, color='0.6')
ax[0].axvline(V_HOVER, color='k', label=f'V_hover {V_HOVER:.3f}')
ax[0].set_xlim(0, 1)
ax[0].set_xlabel('mean throttle of the four motors')
ax[0].set_title('the hover within the range 0 to 1')
ax[0].legend()
ax[1].plot(vbat, V_all, '.', ms=3, alpha=0.5)
vv = np.linspace(vbat.min(), vbat.max(), 2)
ax[1].plot(vv, slope * vv + intercept, 'k', label=f'{slope:+.3f} per V')
ax[1].set_xlabel('battery [V]')
ax[1].set_ylabel('mean throttle')
ax[1].set_title('hover throttle against the battery')
ax[1].legend()
for a in ax:
    a.grid()
plt.tight_layout()

## Step 9: record the calibration

The cell prints the numbers in the form the simulator uses and adds them as one line to `data/motor_calibration.csv`, with the drone, the time and the battery, so that the calibrations of several flights, batteries or drones can be compared.

In [ ]:
print(f'V_hover       = {V_HOVER:.4f}')
print(f'V_left_offset = {V_LEFT_OFFSET:+.4f}')
print(f'V_front_offset= {V_FRONT_OFFSET:+.4f}   (no counterpart in the 2D simulator)')
print(f'V_yaw_offset  = {V_YAW_OFFSET:+.4f}   (no counterpart in the 2D simulator)')

path = os.path.join('data', 'motor_calibration.csv')
os.makedirs('data', exist_ok=True)
new_file = not os.path.exists(path)
with open(path, 'a') as f:
    if new_file:
        f.write('run,drone,vbat_start,vbat_end,V_hover,V_left_offset,V_front_offset,V_yaw_offset,m1,m2,m3,m4\n')
    f.write(f'{name},{r["drone"]},{vbat.max():.3f},{vbat.min():.3f},{V_HOVER:.4f},{V_LEFT_OFFSET:.4f},'
            f'{V_FRONT_OFFSET:.4f},{V_YAW_OFFSET:.4f},' + ','.join(f'{means[m]:.4f}' for m in (1, 2, 3, 4)) + '\n')
print('added to', path)

## Questions

* Fly the calibration twice in a row. How close are the two `V_hover`, and the two `V_left_offset`? Is the difference larger than the scatter the cell printed?
* Fly it once more when the battery is down to about 3.6 V at rest. Does `V_hover` change, as the slope in step 8 predicted?
* Move the battery a few millimetres towards the front, or fly without the cage. Which offset moves, and which way? Does it agree with the layout of the motors at the top?
* Compare `V_hover` and the offsets with the values you found for the simulated drone. What does the 2D simulator leave out that the real drone needs?
* The firmware's controller found the trim by itself, with its integrators. Your own controllers in the simulator did not have that: how did you handle `V_left_offset` there, and what would happen with an offset you got wrong?

## If something goes wrong

* `this run has no hover phase`: the flight was aborted before the hold. Check the abort reason printed by the flight cell and fly again.
* All four motors read 0: the log of the power group did not arrive. Run the status cell, then fly again.
* The motor lines swing up and down in the grey band: the drone did not hover steadily (draughts, a floor with little texture, a low battery). Fly over the mat, away from fans and doors.
* `No drone answers at ...`, `is locked: yes`, `state samples stale`, `height sensor frozen`: as in the other flight notebooks; status, recover, switch off and on.